In [ ]:
import pandas as pd
from typing import Tuple, Optional


In [ ]:
data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results"

results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results"

scratch = "/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results"

#!mkdir {scratch}

In [ ]:
#import aux file functions

import pandas as pd

def load_ancestry(path: str = f"{data}/ancestry.csv") -> pd.DataFrame:
    return pd.read_csv(path)

def load_demo(path: str = f"{data}/demographics_table.csv") -> pd.DataFrame:
    return pd.read_csv(path)

def load_flagged(path: str = f"{data}/flagged_samples.tsv") -> pd.DataFrame:
    # assuming tab-separated file
    return pd.read_csv(path, sep="\t")

def load_phex(path: str = f"{data}/phecodex_info.csv") -> pd.DataFrame:
    return pd.read_csv(path)

def load_related(path: str = f"{data}/relatedness_flagged_samples.tsv") -> pd.DataFrame:
    # assuming tab-separated file
    return pd.read_csv(path, sep="\t")

def load_phex_map(path: str = f"{data}/updated_phecodex_map.csv") -> pd.DataFrame:
    return pd.read_csv(path)

def load_srwgs_samples(path: str = f"{data}/v7_has_srWGS.csv") -> pd.DataFrame:
    return pd.read_csv(path)




In [ ]:
#import pcs
def flatten_ancestry_pcs_df(pc_file): 
    
    pc_df = pd.read_csv(pc_file, sep='\t')
    
    pc_df["pca_features"] = pc_df["pca_features"].str[1:-1]
    
    PCs = pc_df["pca_features"].str.split(",", n = 16, expand = True)
    PCs = PCs.astype(float)
    
    pid = pc_df[["research_id"]]
    
    columns= ["PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10","PC11","PC12","PC13","PC14","PC15","PC16"]
    PCs.columns = columns
    
    PCs_final = pd.concat([pid, PCs], axis = 1)
    
    PCs_final.to_csv('wrangled_ancestry_pcs.csv')
    
    return PCs_final

In [ ]:
#import phecode (== MS_700.11)

def get_case_and_control_df(
    csv_path: str,
    phecode: str,
    sex: Optional[str] = None,   # "M" / "F" or None for both
    sex_col: str = "sex",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    For a single phecode, return:
      - cases_df:    rows where phecode == True
      - controls_df: rows where phecode == False

    Only person_id, that phecode, and sex are read from the big CSV.
    NaN in the phecode column = out of risk set → excluded.
    """

    # Only load the columns we need from disk
    usecols = ["person_id", phecode, sex_col]
    df = pd.read_csv(csv_path, usecols=usecols)

    # valid rows for this phecode
    mask_valid = df[phecode].notna()

    # optional sex restriction
    if sex is not None:
        mask_valid &= df[sex_col] == sex

    # Cases: True
    mask_cases = mask_valid & (df[phecode] == True)

    # Controls: False
    mask_controls = mask_valid & (df[phecode] == False)

    cols_out = ["person_id", sex_col]

    cases_df = df.loc[mask_cases, cols_out].reset_index(drop=True)
    controls_df = df.loc[mask_controls, cols_out].reset_index(drop=True)

    return cases_df, controls_df

In [ ]:
def remove_flagged_and_related_samples(flag_df, related_df, df_case, df_cont):
    # Collect IDs to drop
    related_ids = set(related_df["sample_id"])
    flagged_ids = set(flag_df["s"])
    ids_to_drop = related_ids | flagged_ids  # union

    def _filter(df):
        return df[~df["person_id"].isin(ids_to_drop)].reset_index(drop=True)

    df_final_case = _filter(df_case)
    df_final_cont = _filter(df_cont)

    return df_final_case, df_final_cont

In [ ]:
#merge filtered case and control files with demo and ancestry data

def add_demo_and_ancesry_data(demo, ancestry,  fil_df_case, fil_df_cont):
    

    demo_case = pd.merge(fil_df_case, demo, on = "person_id", how = "left")
    final_case = pd.merge(demo_case, ancestry, on = "person_id", how = "left")
    
    demo_cont = pd.merge(fil_df_cont, demo, on = "person_id", how = "left")
    final_cont = pd.merge(demo_cont, ancestry, on = "person_id", how = "left")
    
    return final_case, final_cont
    

    


In [ ]:
def add_pcs_to_covar_files_and_filter_for_srwgs(pcs, srwgs, case, control):
    
    pcs = pcs.rename(columns={"research_id": "person_id"})
    
    case_pcs = pd.merge(case, pcs, on = "person_id", how = "left")
    cont_pcs = pd.merge(control, pcs, on = "person_id", how = "left")
    
    sr_samples = srwgs["person_id"]
    
    final_case = case_pcs[case_pcs["person_id"].isin(sr_samples)].reset_index(drop=True)
    final_cont = cont_pcs[cont_pcs["person_id"].isin(sr_samples)].reset_index(drop=True)
    
    return final_case, final_cont


In [ ]:
def add_phenotype_col_and_filter_on_sex_and_age(case, control):
    
    
    #cases == 1
    #control == 0
    
    case["case"] = 1
    control["case"] = 0
    
    #filter on age <=100
    
    case = case[case["age_at_cdr"] <= 100]
    control = control[control["age_at_cdr"] <= 100]
    
    #remove samples who dont have sex
    
    sex = ["M", "F"]
   
    final_case = case[case["sex"].isin(sex)].reset_index(drop=True)
    final_cont = control[control["sex"].isin(sex)].reset_index(drop=True)
    
    return final_case, final_cont
    
    

In [ ]:
def get_missing_covar_cols (case, control):
    
    case = case.copy()
    control = control.copy()
    
    
    #convert sex to 0 = male, and 1 = female
   
    case["sex"] = case["sex"].map({"M": 0, "F": 1})
    control["sex"] = control["sex"].map({"M": 0, "F": 1})

    #add age*sex, age^2, age^2*sex, 
    
    case["age2"] = case["age_at_cdr"]**2
    control["age2"] = control["age_at_cdr"]**2

    case["age_sex"] = case["sex"] * case["age_at_cdr"]
    control["age_sex"] = control["sex"] * control["age_at_cdr"]
    
    case["age2_sex"] = case["sex"] * case["age2"]
    control["age2_sex"] = control["sex"] * control["age2"]
    
    
    return case, control 

In [ ]:
def get_concat_pheno_covar_files_for_case_controls(case, control, filename):
    
    final = pd.concat([case, control], axis = 0,  ignore_index=True)
    
    final = final.rename(columns={"age_at_cdr": "age"})
    
    #filter for F only
    
    final = final[final["sex"] == 1]
    
    final.to_csv(filename)
    
    return final

In [ ]:
#now correcly format matchit files for regenie input

In [ ]:
import pandas as pd

def format_pheno_covar_merge_file(filename, filename2):
    # Original CSV


    df = pd.read_csv(f"{results}/{filename}")

    # Drop the index column `...1` if present
    if "...1" in df.columns:
        df = df.drop(columns=["...1"])

    # Create FID/IID from person_id
    df["FID"] = df["person_id"]
    df["IID"] = df["person_id"]

    # Columns you want for REGENIE
    covars = ["sex", "age", "age2", "age_sex", "age2_sex"] + [f"PC{i}" for i in range(1, 17)]
    pheno_col = "case"

    cols = ["FID", "IID", pheno_col] + covars

    # Check all are present
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}")

    df_out = df[cols]

    # Write whitespace-delimited file for REGENIE
    df_out.to_csv(f"{results}/{filename2}", sep="\t", index=False)

    print("Wrote REGENIE pheno/covar file to:", f"{results}/{filename2}")
    print(df_out.head())
    
    return df_out

In [ ]:
def filter_for_final_qc_samples(filename):
    
    df1 = pd.read_csv("lupus_matched_ALL.regenie.txt", sep="\t")
    
    samples = pd.read_csv(f"{data}/sample_ids.tsv", sep="\t")
    qc_list = set(samples["person_id"])
    
    final = df1[df1["IID"].isin(qc_list)].reset_index(drop=True)
    
    # Write whitespace-delimited file for REGENIE
    final.to_csv(f"{results}/{filename}", sep="\t", index=False)
    
    return final
    

In [ ]:
#function calls
'''
ancestry = load_ancestry()
demo = load_demo()
flagged = load_flagged()
phex = load_phex()
related = load_related()
phex_map = load_phex_map()
pcs = flatten_ancestry_pcs_df(f"{data}/ancestry_preds.tsv")
srwgs = load_srwgs_samples()

lupus, lupus_cont  = get_case_and_control_df(f"{data}/mcc2_phecodex_table.csv", "MS_700.11")


lupus_flag, lupus_cont_flag = remove_flagged_and_related_samples(flagged, related, lupus, lupus_cont)



lupus_df, lupus_cont_df = add_demo_and_ancesry_data(demo, ancestry, lupus_flag, lupus_cont_flag)


lupus_phenos, lupus_cont_phenos = add_pcs_to_covar_files_and_filter_for_srwgs(pcs,srwgs, lupus_df, lupus_cont_df)



lupus2, lupus2_cont = add_phenotype_col_and_filter_on_sex_and_age(lupus_phenos, lupus_cont_phenos)



lupus3, lupus3_cont = get_missing_covar_cols (lupus2, lupus2_cont)




final_lupus = get_concat_pheno_covar_files_for_case_controls(lupus3, lupus3_cont, "lupus_matchit.csv")
'''

#df = format_pheno_covar_merge_file("lupus_matched_ALL.csv", "lupus_matched_ALL.regenie.txt")
df2 = filter_for_final_qc_samples("lupus_matched_ALL_final.regenie.txt")

In [ ]:
#visualize

#ancestry 
#demo
#flagged
#phex 
#related 
#phex_map
#pcs
#hepc
#hepc_controls 
#hepb
#hepb_controls   
#viral
#viral_controls

#lupus
#lupus_cont  


#lupus_flag
#lupus_cont_flag 

#lupus_df
#lupus_cont_df


#lupus_phenos
#lupus_cont_phenos 



#lupus2
#lupus2_cont



#lupus3
#lupus3_cont 

final_lupus


In [ ]:
final_lupus[final_lupus["case"] == 0]

In [ ]:
# Number of missing values per column
missing_counts = viral3_cont.isna().sum()

# Column with the most missing values
most_missing_col = missing_counts.idxmax()
most_missing_count = missing_counts.max()

print("Column with most missing values:", most_missing_col)
print("Number of missing values:", most_missing_count)

# If you want to see all columns sorted by missingness:
print(missing_counts.sort_values(ascending=False))

In [ ]:
df5 = final_hepc[final_hepc["case"]== 1]

In [ ]:
df5

In [ ]:
pwd

In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/hepb_matched_10.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/hepb_matched_ALL.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/hepc_matched_ALL.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/hepc_matched_10.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/viral_matched_10.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/viral_matched_ALL.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  
